# Penyusunan Transkrip Acuan — Analisis dan Perancangan Sistem, Interaksi Manusia dan Komputer, Arsitektur dan Organisasi Komputer

Notebook ini menjalankan langkah subbab **3.3.1** (penyaringan dan pengumpulan), **3.6.1** (praproses audio), dan **3.3.2** (penyusunan transkrip acuan) sampai menghasilkan master transkrip acuan berformat TSV.

## Alur

```
[1] Manifest rekaman            -> data/output/manifest_rekaman.csv
[2] Unduh audio (yt-dlp)        -> data/raw/audio_src/
[3] Praproses audio (ffmpeg)    -> data/processed/audio/     16 kHz, mono, PCM 16-bit
[4] Transkripsi large-v3        -> data/raw/json/            arsip mentah, tak disunting
[5] Prioritas simak             -> data/output/review_priority.csv
[6] Ekspor SRT                  -> data/processed/srt/
    ============ KOREKSI MANUAL DI SUBTITLE EDIT ============
    Simpan hasil ke              data/processed/srt_done/
[7] SRT terkoreksi -> TSV       -> data/output/master/
[8] Validasi
[9] Glosarium
[10] Ringkasan untuk Bab 4
```

Sel [1]–[6] dijalankan sekali, lalu berhenti. Sel [7] ke bawah dijalankan setelah koreksi manual selesai.

> **Catatan.** Transkrip acuan dikoreksi di **SRT**, bukan di TSV. Pedoman 3.3.2 menuntut koreksi dijalankan sambil menyimak audio penuh, karena penghapusan kata berulang dan halusinasi hanya tertangkap dengan cara itu. TSV adalah format simpan, bukan permukaan kerja.

## 0. Persiapan

In [ ]:
# Jalankan sekali bila pustaka belum terpasang.
# !pip install faster-whisper yt-dlp pandas srt tqdm
# FFmpeg harus tersedia di PATH (cek: !ffmpeg -version)

In [1]:
from pathlib import Path
import json, re, subprocess, shutil, unicodedata, datetime
import pandas as pd

# Notebook berada di notebooks/ , sehingga akar proyek satu tingkat di atas.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW        = ROOT / "data" / "raw"
PROC       = ROOT / "data" / "processed"
OUT        = ROOT / "data" / "output"

AUDIO_SRC  = RAW  / "audio_src"      # hasil unduh, apa adanya
JSON_RAW   = RAW  / "json"           # keluaran large-v3, arsip
AUDIO_PROC = PROC / "audio"          # 16 kHz mono PCM16
SRT_TODO   = PROC / "srt"            # untuk dikoreksi
SRT_DONE   = PROC / "srt_done"       # hasil koreksi manual
MASTER     = OUT  / "master"         # TSV transkrip acuan

for d in (AUDIO_SRC, JSON_RAW, AUDIO_PROC, SRT_TODO, SRT_DONE, MASTER, OUT):
    d.mkdir(parents=True, exist_ok=True)

print("Akar proyek :", ROOT)
print("FFmpeg      :", shutil.which("ffmpeg") or "TIDAK DITEMUKAN")

Akar proyek : c:\Users\Aufii\Documents\Informatika\OTHERS\8. RISET-SKRIPSI\assistive-communicative
FFmpeg      : C:\Users\Aufii\AppData\Local\Microsoft\WinGet\Links\ffmpeg.EXE


In [2]:
# ---------------------------------------------------------------------------
# Konfigurasi. Nilai decoding mengikuti Tabel 3.5.
# ---------------------------------------------------------------------------
CFG = {
    "model_acuan": "large-v3",          # model penyusun transkrip awal (3.3.2)
    "device": "cuda",
    "compute_type": "float16",          # samakan antarvarian agar RTF sebanding
    "language": "id",                   # ditetapkan, bukan dideteksi
    "task": "transcribe",
    "beam_size": 5,
    "temperature": [0.0, 0.2, 0.4, 0.6, 0.8, 1.0],   # fallback bertahap
    "vad_filter": True,
    "condition_on_previous_text": False,
    "word_timestamps": False,
    "sample_rate": 16000,
    "channels": 1,
    "loudnorm_I": -23.0,                # EBU R128, sasaran lantang terpadu
    "loudnorm_TP": -2.0,
    "loudnorm_LRA": 7.0,
}
CFG

{'model_acuan': 'large-v3',
 'device': 'cuda',
 'compute_type': 'float16',
 'language': 'id',
 'task': 'transcribe',
 'beam_size': 5,
 'temperature': [0.0, 0.2, 0.4, 0.6, 0.8, 1.0],
 'vad_filter': True,
 'condition_on_previous_text': False,
 'word_timestamps': False,
 'sample_rate': 16000,
 'channels': 1,
 'loudnorm_I': -23.0,
 'loudnorm_TP': -2.0,
 'loudnorm_LRA': 7.0}

## 1. Manifest rekaman

Kolom `ada_takarir_manusia` memenuhi kriteria penyaringan nomor 2 pada subbab 3.3.1. Isi `False` hanya setelah dicek di halaman videonya (Settings → Subtitles: yang tertera *auto-generated* boleh, yang tertera bahasa tanpa keterangan itu buatan manusia dan rekamannya harus dikeluarkan).

Kolom `split` dikosongkan dulu; pembagiannya dikerjakan pada subbab 3.6.2.

In [3]:
REKAMAN = [
    # (file_id, mata_kuliah, url, durasi_tertera)
    ("APS-01", "APS", "https://www.youtube.com/watch?v=HOG8CUse5HY", "13:18"),
    ("APS-02", "APS", "https://www.youtube.com/watch?v=6OU9phN5dDQ", "11:08"),
    ("APS-03", "APS", "https://www.youtube.com/watch?v=k1z9gy1fVy8", "10:42"),
    ("APS-04", "APS", "https://www.youtube.com/watch?v=OMliDo_CpxE", "12:19"),
    ("APS-05", "APS", "https://www.youtube.com/watch?v=ToESrvlgg3o", "9:41"),
    ("APS-06", "APS", "https://www.youtube.com/watch?v=ikke0VwiQS0", "20:41"),

    ("IMK-01", "IMK", "https://www.youtube.com/watch?v=VWIuwAQZWSY", "15:41"),
    ("IMK-02", "IMK", "https://www.youtube.com/watch?v=nh-xG-v8whc", "18:42"),
    ("IMK-03", "IMK", "https://www.youtube.com/watch?v=aS_dB3dvWIU", "15:48"),
    ("IMK-04", "IMK", "https://www.youtube.com/watch?v=TNr8swEdr9g", "16:36"),
    ("IMK-05", "IMK", "https://www.youtube.com/watch?v=mC_PPEjBLTs", "17:30"),

    ("AOK-01", "AOK", "https://www.youtube.com/watch?v=SAjBsxQFV8w", "14:17"),
    ("AOK-02", "AOK", "https://www.youtube.com/watch?v=-a9iZ8y999Y", "34:42"),
    ("AOK-03", "AOK", "https://www.youtube.com/watch?v=6pUbgRsESkc", "12:48"),
    ("AOK-04", "AOK", "https://www.youtube.com/watch?v=mkdzKqmMxto", "19:56"),
    ("AOK-05", "AOK", "https://www.youtube.com/watch?v=sFFBdaQj_J0", "18:05"),
    ("AOK-06", "AOK", "https://www.youtube.com/watch?v=6NVTqNtVrCw", "15:32"),
    ("AOK-07", "AOK", "https://www.youtube.com/watch?v=TCAx_b8fWF0", "15:58"),
    ("AOK-08", "AOK", "https://www.youtube.com/watch?v=JLxEKYbm_c8", "13:48"),
]

NAMA_MK = {
    "APS": "Analisis dan Perancangan Sistem",
    "IMK": "Interaksi Manusia dan Komputer",
    "AOK": "Arsitektur dan Organisasi Komputer",
}

def mmss_ke_detik(s):
    b = [float(x) for x in s.split(":")]
    return b[0] * 60 + b[1] if len(b) == 2 else b[0] * 3600 + b[1] * 60 + b[2]

MANIFEST_PATH = OUT / "manifest_rekaman.csv"

if MANIFEST_PATH.exists():
    manifest = pd.read_csv(MANIFEST_PATH, dtype=str)
    print("Manifest lama dimuat, tidak ditimpa.")
else:
    manifest = pd.DataFrame([{
        "file_id": fid,
        "mata_kuliah": mk,
        "nama_mata_kuliah": NAMA_MK[mk],
        "url": url,
        "durasi_tertera": dur,
        "durasi_tertera_detik": mmss_ke_detik(dur),
        "durasi_asli_detik": "",        # diisi sel [3]
        "durasi_potong_detik": "",      # diisi sel [3]
        "trim_awal_detik": 0,           # isi manual bila ada pembuka bukan materi
        "trim_akhir_detik": 0,          # isi manual bila ada penutup bukan materi
        "ada_takarir_manusia": "",      # WAJIB diisi manual: True / False
        "lolos_penyaringan": "",        # True / False + alasan di kolom catatan
        "split": "",                    # latih / validasi / uji  (subbab 3.6.2)
        "tanggal_unduh": "",
        "catatan": "",
    } for fid, mk, url, dur in REKAMAN])
    manifest.to_csv(MANIFEST_PATH, index=False, encoding="utf-8-sig")
    print("Manifest dibuat:", MANIFEST_PATH)

tot = manifest["durasi_tertera_detik"].astype(float).sum()
print(f"\n{len(manifest)} rekaman, total {tot/3600:.2f} jam")
print(manifest.groupby("mata_kuliah")["durasi_tertera_detik"]
              .agg(jumlah="count", detik="sum")
              .assign(jam=lambda d: (d["detik"]/3600).round(2)))

Manifest dibuat: c:\Users\Aufii\Documents\Informatika\OTHERS\8. RISET-SKRIPSI\assistive-communicative\data\output\manifest_rekaman.csv

19 rekaman, total 5.12 jam
             jumlah   detik   jam
mata_kuliah                      
AOK               8  8706.0  2.42
APS               6  4669.0  1.30
IMK               5  5057.0  1.40


## 2. Unduh audio

Hanya jalur audio yang diunduh. Berkas video tidak diperlukan dan hanya memakan ruang.

In [5]:
def unduh(file_id, url, force=False):
    tujuan = AUDIO_SRC / f"{file_id}.m4a"
    if tujuan.exists() and not force:
        return tujuan, "sudah ada"
    perintah = [
        "yt-dlp", "-f", "bestaudio[ext=m4a]/bestaudio",
        "--extract-audio", "--audio-format", "m4a", "--audio-quality", "0",
        "--no-playlist", "--no-write-subs", "--no-write-auto-subs",
        "-o", str(AUDIO_SRC / f"{file_id}.%(ext)s"),
        url,
    ]
    hasil = subprocess.run(perintah, capture_output=True, text=True)
    if hasil.returncode != 0:
        return None, hasil.stderr.strip().splitlines()[-1] if hasil.stderr else "gagal"
    return tujuan, "terunduh"

hari_ini = datetime.date.today().isoformat()
for i, baris in manifest.iterrows():
    berkas, status = unduh(baris["file_id"], baris["url"])
    print(f"{baris['file_id']:8s} {status}")
    if berkas and not str(manifest.at[i, "tanggal_unduh"]).strip():
        manifest.at[i, "tanggal_unduh"] = hari_ini

manifest.to_csv(MANIFEST_PATH, index=False, encoding="utf-8-sig")

APS-01   sudah ada
APS-02   terunduh
APS-03   sudah ada
APS-04   sudah ada
APS-05   sudah ada
APS-06   sudah ada
IMK-01   sudah ada
IMK-02   terunduh
IMK-03   sudah ada
IMK-04   sudah ada
IMK-05   terunduh
AOK-01   terunduh
AOK-02   terunduh
AOK-03   terunduh
AOK-04   terunduh
AOK-05   terunduh
AOK-06   terunduh
AOK-07   terunduh
AOK-08   terunduh


## 3. Praproses audio (subbab 3.6.1)

Tiga hal dikerjakan sekaligus: penyeragaman format ke 16 kHz / mono / PCM 16-bit mengikuti laju cuplik masukan Whisper, normalisasi lantang dengan EBU R128, dan pemotongan bagian pembuka serta penutup yang bukan materi kuliah.

Nilai `trim_awal_detik` dan `trim_akhir_detik` diisi manual di manifest sebelum sel ini dijalankan. Simak beberapa detik pertama dan terakhir tiap rekaman untuk menentukannya. Bila tidak ada yang perlu dipotong, biarkan 0.

Penyaring VAD **tidak** dijalankan di sini. Segmen tanpa wicara tetap ikut menjadi penyebut RTF, sesuai subbab 3.6.8.

In [7]:
def durasi_detik(berkas):
    hasil = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration",
         "-of", "default=noprint_wrappers=1:nokey=1", str(berkas)],
        capture_output=True, text=True)
    return float(hasil.stdout.strip()) if hasil.returncode == 0 else None

def praproses(file_id, trim_awal=0.0, trim_akhir=0.0, force=False):
    sumber = AUDIO_SRC / f"{file_id}.m4a"
    tujuan = AUDIO_PROC / f"{file_id}.wav"
    if not sumber.exists():
        return None, None, "sumber tidak ada"
    if tujuan.exists() and not force:
        return durasi_detik(sumber), durasi_detik(tujuan), "sudah ada"

    d_asli = durasi_detik(sumber)
    perintah = ["ffmpeg", "-y", "-hide_banner", "-loglevel", "error"]
    if trim_awal > 0:
        perintah += ["-ss", str(trim_awal)]
    perintah += ["-i", str(sumber)]
    if trim_akhir > 0:
        perintah += ["-t", str(max(0.0, d_asli - trim_awal - trim_akhir))]
    perintah += [
        "-af", f"loudnorm=I={CFG['loudnorm_I']}:TP={CFG['loudnorm_TP']}:LRA={CFG['loudnorm_LRA']}",
        "-ar", str(CFG["sample_rate"]),
        "-ac", str(CFG["channels"]),
        "-c:a", "pcm_s16le",
        "-map_metadata", "-1",
        str(tujuan),
    ]
    hasil = subprocess.run(perintah, capture_output=True, text=True)
    if hasil.returncode != 0:
        return d_asli, None, hasil.stderr.strip()[:200]
    return d_asli, durasi_detik(tujuan), "diproses"

for i, baris in manifest.iterrows():
    ta = float(baris.get("trim_awal_detik") or 0)
    tk = float(baris.get("trim_akhir_detik") or 0)
    d_asli, d_potong, status = praproses(baris["file_id"], ta, tk)
    if d_asli   is not None: manifest.at[i, "durasi_asli_detik"]   = f"{d_asli:.3f}"
    if d_potong is not None: manifest.at[i, "durasi_potong_detik"] = f"{d_potong:.3f}"
    print(f"{baris['file_id']:8s} {status:12s} "
          f"{'' if d_asli is None else f'{d_asli:8.1f}s'} -> "
          f"{'' if d_potong is None else f'{d_potong:8.1f}s'}")

manifest.to_csv(MANIFEST_PATH, index=False, encoding="utf-8-sig")

APS-01   sudah ada       798.6s ->    798.6s
APS-02   diproses        668.4s ->    668.4s
APS-03   diproses        642.4s ->    642.4s
APS-04   diproses        739.8s ->    739.8s
APS-05   diproses        581.5s ->    581.5s
APS-06   diproses       1213.8s ->   1213.8s
IMK-01   diproses        941.5s ->    941.5s
IMK-02   diproses       1122.5s ->   1122.5s
IMK-03   diproses        948.7s ->    948.7s
IMK-04   diproses        996.4s ->    996.4s
IMK-05   diproses       1050.2s ->   1050.2s
AOK-01   diproses        857.2s ->    857.2s
AOK-02   diproses       2082.6s ->   2082.6s
AOK-03   diproses        768.7s ->    768.7s
AOK-04   diproses       1197.0s ->   1197.0s
AOK-05   diproses       1085.8s ->   1085.8s
AOK-06   diproses        932.3s ->    932.3s
AOK-07   diproses        958.1s ->    958.1s
AOK-08   diproses        828.9s ->    828.9s


In [8]:
periksa = []
for w in sorted(AUDIO_PROC.glob("*.wav")):
    h = subprocess.run(
        ["ffprobe", "-v", "error", "-select_streams", "a:0",
         "-show_entries", "stream=sample_rate,channels,sample_fmt,codec_name",
         "-of", "json", str(w)], capture_output=True, text=True)
    s = json.loads(h.stdout)["streams"][0]
    periksa.append({"file_id": w.stem, **s})

df_periksa = pd.DataFrame(periksa)
print(df_periksa.to_string(index=False))
print("\nSeragam:", df_periksa[["sample_rate", "channels", "sample_fmt"]].drop_duplicates().shape[0] == 1)

file_id codec_name sample_fmt sample_rate  channels
 AOK-01  pcm_s16le        s16       16000         1
 AOK-02  pcm_s16le        s16       16000         1
 AOK-03  pcm_s16le        s16       16000         1
 AOK-04  pcm_s16le        s16       16000         1
 AOK-05  pcm_s16le        s16       16000         1
 AOK-06  pcm_s16le        s16       16000         1
 AOK-07  pcm_s16le        s16       16000         1
 AOK-08  pcm_s16le        s16       16000         1
 APS-01  pcm_s16le        s16       16000         1
 APS-02  pcm_s16le        s16       16000         1
 APS-03  pcm_s16le        s16       16000         1
 APS-04  pcm_s16le        s16       16000         1
 APS-05  pcm_s16le        s16       16000         1
 APS-06  pcm_s16le        s16       16000         1
 IMK-01  pcm_s16le        s16       16000         1
 IMK-02  pcm_s16le        s16       16000         1
 IMK-03  pcm_s16le        s16       16000         1
 IMK-04  pcm_s16le        s16       16000         1
 IMK-05  pcm

## 4. Transkripsi awal dengan large-v3 (subbab 3.3.2)

Keluaran disimpan sebagai JSON dan **tidak pernah disunting**. Berkas ini jadi jejak audit sekaligus sumber kolom `teks_asr` pada master, yang memungkinkan pelaporan seberapa banyak transkrip yang dikoreksi.

Selain teks dan stempel waktu, tiga besaran ikut disimpan: `avg_logprob`, `no_speech_prob`, dan `compression_ratio`. Ketiganya dipakai menyusun prioritas simak pada sel berikutnya.

In [9]:
from faster_whisper import WhisperModel

model = WhisperModel(CFG["model_acuan"], device=CFG["device"], compute_type=CFG["compute_type"])
print("Model dimuat:", CFG["model_acuan"], "|", CFG["device"], CFG["compute_type"])

c:\Users\Aufii\Documents\Informatika\OTHERS\8. RISET-SKRIPSI\assistive-communicative\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model dimuat: large-v3 | cuda float16


In [10]:
import time

def transkripsi(file_id, force=False):
    audio = AUDIO_PROC / f"{file_id}.wav"
    tujuan = JSON_RAW / f"{file_id}.json"
    if tujuan.exists() and not force:
        return "sudah ada"
    if not audio.exists():
        return "audio tidak ada"

    t0 = time.perf_counter()
    segmen_iter, info = model.transcribe(
        str(audio),
        language=CFG["language"],
        task=CFG["task"],
        beam_size=CFG["beam_size"],
        temperature=CFG["temperature"],
        vad_filter=CFG["vad_filter"],
        condition_on_previous_text=CFG["condition_on_previous_text"],
        word_timestamps=CFG["word_timestamps"],
    )
    segmen = [{
        "id": s.id, "start": round(s.start, 3), "end": round(s.end, 3),
        "text": s.text.strip(),
        "avg_logprob": s.avg_logprob,
        "no_speech_prob": s.no_speech_prob,
        "compression_ratio": s.compression_ratio,
        "temperature": s.temperature,
    } for s in segmen_iter]
    lama = time.perf_counter() - t0

    tujuan.write_text(json.dumps({
        "file_id": file_id,
        "model": CFG["model_acuan"],
        "compute_type": CFG["compute_type"],
        "parameter": {k: CFG[k] for k in
                      ("language", "task", "beam_size", "temperature",
                       "vad_filter", "condition_on_previous_text", "word_timestamps")},
        "durasi_audio": info.duration,
        "durasi_setelah_vad": getattr(info, "duration_after_vad", None),
        "waktu_proses": round(lama, 3),
        "jumlah_segmen": len(segmen),
        "segments": segmen,
    }, ensure_ascii=False, indent=2), encoding="utf-8")
    return f"{len(segmen)} segmen, {lama:.0f}s"


for fid in manifest["file_id"]:
    print(f"{fid:8s} {transkripsi(fid)}")

APS-01   165 segmen, 92s
APS-02   128 segmen, 104s
APS-03   122 segmen, 198s
APS-04   172 segmen, 204s
APS-05   231 segmen, 148s
APS-06   218 segmen, 483s
IMK-01   190 segmen, 380s
IMK-02   207 segmen, 461s
IMK-03   171 segmen, 381s
IMK-04   170 segmen, 317s
IMK-05   372 segmen, 490s
AOK-01   320 segmen, 295s
AOK-02   653 segmen, 613s
AOK-03   236 segmen, 229s
AOK-04   512 segmen, 307s
AOK-05   411 segmen, 288s
AOK-06   335 segmen, 302s
AOK-07   355 segmen, 292s
AOK-08   346 segmen, 290s


## 5. Prioritas simak

Segmen dengan rasio kompresi tinggi, peluang tanpa-wicara tinggi, atau log-probabilitas rendah adalah kandidat halusinasi dan pengulangan. Koenecke, et al. (2024) mengaitkan durasi non-vokal panjang dengan munculnya halusinasi, sehingga jeda panjang antarsegmen ikut ditandai.

Daftar ini **bukan pengganti simak penuh**. Pedoman 3.3.2 tetap menuntut seluruh audio disimak. Daftar ini hanya menandai baris yang perlu perhatian ekstra.

In [11]:
AMBANG = {
    "compression_ratio": 2.4,   # di atas ini: teks berulang
    "no_speech_prob": 0.60,     # di atas ini: kemungkinan tanpa wicara
    "avg_logprob": -1.00,       # di bawah ini: model tidak yakin
    "jeda_detik": 3.0,          # jeda sebelum segmen ini
    "durasi_segmen": 28.0,      # mendekati batas 30 detik
}

def ngram_berulang(teks, n=4):
    kata = teks.lower().split()
    if len(kata) < n * 2:
        return False
    gram = [" ".join(kata[i:i+n]) for i in range(len(kata)-n+1)]
    return len(gram) - len(set(gram)) >= 2

baris_prioritas = []
for jf in sorted(JSON_RAW.glob("*.json")):
    d = json.loads(jf.read_text(encoding="utf-8"))
    prev_end = 0.0
    for s in d["segments"]:
        alasan = []
        if s["compression_ratio"] > AMBANG["compression_ratio"]: alasan.append("kompresi-tinggi")
        if s["no_speech_prob"]   > AMBANG["no_speech_prob"]:     alasan.append("mungkin-senyap")
        if s["avg_logprob"]      < AMBANG["avg_logprob"]:        alasan.append("logprob-rendah")
        if s["start"] - prev_end > AMBANG["jeda_detik"]:         alasan.append("jeda-panjang")
        if s["end"] - s["start"] > AMBANG["durasi_segmen"]:      alasan.append("segmen-panjang")
        if ngram_berulang(s["text"]):                            alasan.append("frasa-berulang")
        if alasan:
            baris_prioritas.append({
                "file_id": d["file_id"], "seg_id": s["id"],
                "start": s["start"], "end": s["end"],
                "waktu": f"{int(s['start']//60):02d}:{s['start']%60:05.2f}",
                "alasan": "|".join(alasan),
                "compression_ratio": round(s["compression_ratio"], 2),
                "no_speech_prob": round(s["no_speech_prob"], 3),
                "avg_logprob": round(s["avg_logprob"], 3),
                "teks_asr": s["text"],
                "sudah_disimak": "",
            })
        prev_end = s["end"]

df_prioritas = pd.DataFrame(baris_prioritas)
df_prioritas.to_csv(OUT / "review_priority.csv", index=False, encoding="utf-8-sig")
print(f"{len(df_prioritas)} segmen ditandai")
if len(df_prioritas):
    print(df_prioritas["alasan"].str.split("|").explode().value_counts())

20 segmen ditandai
alasan
jeda-panjang      12
segmen-panjang     7
mungkin-senyap     2
Name: count, dtype: int64


## 6. Ekspor SRT untuk koreksi manual

Berkas SRT ditulis ke `data/processed/srt/`. **Jangan menyunting berkas di folder itu.** Buka di Subtitle Edit, koreksi, lalu simpan ke `data/processed/srt_done/` dengan nama yang sama. Pemisahan folder ini menjaga agar keluaran mentah tetap utuh sebagai pembanding.

In [12]:
def detik_ke_srt(t):
    ms = int(round(t * 1000))
    j, ms = divmod(ms, 3600000)
    m, ms = divmod(ms, 60000)
    d, ms = divmod(ms, 1000)
    return f"{j:02d}:{m:02d}:{d:02d},{ms:03d}"

for jf in sorted(JSON_RAW.glob("*.json")):
    d = json.loads(jf.read_text(encoding="utf-8"))
    blok = []
    for i, s in enumerate(d["segments"], start=1):
        blok.append(f"{i}\n{detik_ke_srt(s['start'])} --> {detik_ke_srt(s['end'])}\n{s['text']}\n")
    (SRT_TODO / f"{d['file_id']}.srt").write_text("\n".join(blok), encoding="utf-8")
    print(f"{d['file_id']:8s} {len(d['segments'])} baris")

print("\nSRT siap dikoreksi di:", SRT_TODO)

AOK-01   320 baris
AOK-02   653 baris
AOK-03   236 baris
AOK-04   512 baris
AOK-05   411 baris
AOK-06   335 baris
AOK-07   355 baris
AOK-08   346 baris
APS-01   165 baris
APS-02   128 baris
APS-03   122 baris
APS-04   172 baris
APS-05   231 baris
APS-06   218 baris
IMK-01   190 baris
IMK-02   207 baris
IMK-03   171 baris
IMK-04   170 baris
IMK-05   372 baris

SRT siap dikoreksi di: c:\Users\Aufii\Documents\Informatika\OTHERS\8. RISET-SKRIPSI\assistive-communicative\data\processed\srt


---

# ⛔ JEDA: koreksi manual

Sel di bawah baru dijalankan setelah seluruh SRT dikoreksi.

## Penyiapan Subtitle Edit

1. Buka `Video → Open video file`, pilih WAV dari `data/processed/audio/` — **bukan** m4a dari `audio_src`, agar stempel waktunya sejalan dengan berkas yang nanti ditranskripsi model uji.
2. Buka SRT dari `data/processed/srt/`.
3. `Options → Settings → Waveform` → aktifkan tampilan gelombang. Bagian senyap panjang langsung terlihat.
4. Matikan koreksi otomatis: `Options → Settings → Tools` → nonaktifkan *Fix common errors on load* dan seluruh penggantian otomatis. Pedoman 3.3.2 menuntut verbatim; perbaikan otomatis akan melanggarnya diam-diam.
5. `Simpan As` ke `data/processed/srt_done/` dengan nama berkas **persis sama**, format `SubRip (.srt)`, encoding **UTF-8**.

Pintasan yang menghemat waktu: `Ctrl+Space` putar baris terpilih, `Alt+↓` baris berikutnya, `Ctrl+Shift+L` pecah baris pada posisi kursor.

## Yang dikoreksi (pedoman subbab 3.3.2)

1. **Verbatim**, tanda baca dan kapitalisasi tetap dituliskan.
2. **Istilah** dicatat dalam bentuk yang benar-benar diucapkan, ejaan baku bahasa asalnya bila istilah asing. Bila bentuk Inggris dan pelafalan Indonesianya tidak terbedakan secara akustik, ikuti bentuk tertulis pada materi ajar.
3. **Bilangan** ditulis dalam bentuk kata sebagaimana diucapkan — `dua puluh`, bukan `20`. Pengubahan ke angka dikerjakan pada tahap normalisasi (3.6.3). Akronim ditulis rapat berhuruf kapital: `UML`, `API`.
4. **Jeda terisi** ditulis apa adanya (`eee`, `emm`). Kata terpotong ditulis sampai bagian yang terdengar lalu diakhiri tanda hubung: `komu-`.
5. **Tanda hubung dan bentuk ulang** mengikuti Tabel 3.4: `kata-kata`, `pandang-dengar`, `di-encode` dipertahankan sebagai satu kata.

Dua jenis kesalahan yang wajib dikejar sambil menyimak, karena tidak terbaca dari teks saja:

- **Penghapusan kata berulang** — Whisper cenderung membuang pengulangan yang benar-benar diucapkan pengajar.
- **Halusinasi** — teks yang tidak ada dalam audio, dan justru terbaca wajar. Periksa `data/output/review_priority.csv` untuk kandidatnya.

## Batas segmen

Batas boleh dipecah atau digabung bila salah tempat. Sel [7] mencocokkan teks acuan ke teks ASR berdasarkan **irisan waktu**, bukan nomor baris, sehingga perubahan struktur segmen aman.

Yang harus dijaga: stempel waktu tetap urut, tidak tumpang tindih, dan tidak ada baris kosong.

---

## 7. SRT terkoreksi → master TSV

Format TSV dipilih, bukan CSV. Transkrip verbatim penuh koma dan itu adalah pemisah CSV; tab tidak pernah muncul dalam ujaran, sehingga ambiguitasnya nol. Excel membaca TSV sama baiknya lewat `Data → From Text/CSV`.

In [ ]:
def baca_srt(path):
    isi = path.read_text(encoding="utf-8-sig").replace("\r\n", "\n").strip()
    pola = re.compile(
        r"(\d+)\s*\n(\d{2}:\d{2}:\d{2}[,.]\d{3})\s*-->\s*(\d{2}:\d{2}:\d{2}[,.]\d{3})\s*\n(.*?)(?=\n\s*\n|\Z)",
        re.S)
    def ke_detik(t):
        t = t.replace(",", ".")
        j, m, d = t.split(":")
        return int(j) * 3600 + int(m) * 60 + float(d)
    hasil = []
    for m_ in pola.finditer(isi):
        teks = re.sub(r"\s*\n\s*", " ", m_.group(4)).strip()
        teks = re.sub(r"<[^>]+>", "", teks)              # buang tag gaya
        teks = unicodedata.normalize("NFC", teks)
        teks = teks.replace("\t", " ")
        teks = re.sub(r" {2,}", " ", teks).strip()
        hasil.append({"start": ke_detik(m_.group(2)), "end": ke_detik(m_.group(3)), "teks": teks})
    return hasil


def asr_beririsan(segmen_asr, awal, akhir, min_iris=0.20):
    "Gabungkan teks ASR mentah yang beririsan waktu dengan segmen acuan."
    potong = []
    for s in segmen_asr:
        iris = min(akhir, s["end"]) - max(awal, s["start"])
        if iris > 0 and iris / max(1e-9, s["end"] - s["start"]) >= min_iris:
            potong.append(s["text"])
    return " ".join(potong).strip()

In [ ]:
def bangun_master(file_id, force=False):
    srt_path  = SRT_DONE / f"{file_id}.srt"
    json_path = JSON_RAW / f"{file_id}.json"
    tsv_path  = MASTER / f"{file_id}.tsv"

    if not srt_path.exists():
        return None, "SRT terkoreksi belum ada"
    if tsv_path.exists() and not force:
        return pd.read_csv(tsv_path, sep="\t", dtype=str, keep_default_na=False), "sudah ada"

    acuan = baca_srt(srt_path)
    asr   = json.loads(json_path.read_text(encoding="utf-8"))["segments"]
    mk    = manifest.loc[manifest["file_id"] == file_id, "mata_kuliah"].iloc[0]

    baris = []
    for i, a in enumerate(acuan, start=1):
        baris.append({
            "file_id": file_id,
            "mata_kuliah": mk,
            "seg_id": i,
            "start": f"{a['start']:.3f}",
            "end": f"{a['end']:.3f}",
            "dur": f"{a['end'] - a['start']:.3f}",
            "teks_asr": asr_beririsan(asr, a["start"], a["end"]),
            "teks_acuan": a["teks"],
            "flag": "",
            "catatan": "",
        })
    df = pd.DataFrame(baris)
    df.to_csv(tsv_path, sep="\t", index=False, encoding="utf-8-sig")
    return df, f"{len(df)} segmen"


semua = []
for fid in manifest["file_id"]:
    df, status = bangun_master(fid)
    print(f"{fid:8s} {status}")
    if df is not None:
        semua.append(df)

if semua:
    master = pd.concat(semua, ignore_index=True)
    master.to_csv(OUT / "transkrip_acuan_gabungan.tsv", sep="\t",
                  index=False, encoding="utf-8-sig")
    print(f"\nGabungan: {len(master)} segmen dari {master['file_id'].nunique()} rekaman")

## 8. Validasi

Pemeriksaan struktural pada master. Seluruhnya harus lolos sebelum transkrip acuan dipakai.

In [ ]:
def validasi(df):
    masalah = []
    def catat(kode, sub, pesan):
        for _, r in sub.iterrows():
            masalah.append({"file_id": r["file_id"], "seg_id": r["seg_id"],
                            "kode": kode, "pesan": pesan,
                            "cuplikan": str(r["teks_acuan"])[:70]})

    d = df.copy()
    d["start_f"] = d["start"].astype(float)
    d["end_f"]   = d["end"].astype(float)
    d["dur_f"]   = d["end_f"] - d["start_f"]

    catat("KOSONG", d[d["teks_acuan"].astype(str).str.strip() == ""], "teks acuan kosong")
    catat("DUR<=0", d[d["dur_f"] <= 0], "durasi nol atau negatif")
    catat("DUR>30", d[d["dur_f"] > 30.0], "melebihi batas masukan Whisper 30 detik")
    catat("TAB", d[d["teks_acuan"].astype(str).str.contains("\t")], "mengandung tab")
    catat("SPASI_GANDA", d[d["teks_acuan"].astype(str).str.contains("  ")], "spasi ganda")
    catat("KUTIP_TIPOGRAFIS",
          d[d["teks_acuan"].astype(str).str.contains(r"[\u2018\u2019\u201c\u201d]")],
          "tanda kutip melengkung, seragamkan ke lurus")
    catat("ANGKA",
          d[d["teks_acuan"].astype(str).str.contains(r"\d")],
          "mengandung angka; pedoman 3.3.2 poin 3 menuntut bentuk kata")
    catat("HUBUNG_MENGGANTUNG",
          d[d["teks_acuan"].astype(str).str.contains(r"(?:^|\s)-|-(?:\s|$)")],
          "tanda hubung menggantung (sengaja bila kata terpotong)")

    # urutan dan tumpang tindih, per rekaman
    for fid, g in d.groupby("file_id", sort=False):
        g = g.sort_values("seg_id")
        tumpang = g[g["start_f"].values < g["end_f"].shift(1).fillna(-1).values]
        catat("TUMPANG_TINDIH", tumpang, "start lebih awal daripada end segmen sebelumnya")
        if not g["start_f"].is_monotonic_increasing:
            masalah.append({"file_id": fid, "seg_id": "-", "kode": "URUTAN",
                            "pesan": "stempel waktu tidak menaik", "cuplikan": ""})
    return pd.DataFrame(masalah)


if semua:
    lap = validasi(master)
    lap.to_csv(OUT / "laporan_validasi.csv", index=False, encoding="utf-8-sig")
    if len(lap):
        print(lap["kode"].value_counts().to_string())
        print("\nContoh:")
        print(lap.head(15).to_string(index=False))
    else:
        print("Seluruh pemeriksaan lolos.")

In [ ]:
# Seberapa banyak transkrip ASR yang dikoreksi. Angka ini berguna di Bab 4
# sebagai keterangan mutu transkrip acuan.
if semua:
    import difflib
    ringkas = []
    for fid, g in master.groupby("file_id", sort=False):
        a = " ".join(g["teks_asr"].astype(str)).split()
        b = " ".join(g["teks_acuan"].astype(str)).split()
        sm = difflib.SequenceMatcher(None, a, b, autojunk=False)
        sama = sum(blok.size for blok in sm.get_matching_blocks())
        ringkas.append({
            "file_id": fid,
            "kata_asr": len(a),
            "kata_acuan": len(b),
            "kata_diubah": len(b) - sama,
            "persen_diubah": round(100 * (len(b) - sama) / max(1, len(b)), 2),
        })
    df_koreksi = pd.DataFrame(ringkas)
    df_koreksi.to_csv(OUT / "ringkasan_koreksi.csv", index=False, encoding="utf-8-sig")
    print(df_koreksi.to_string(index=False))
    print(f"\nRerata proporsi kata yang dikoreksi: {df_koreksi['persen_diubah'].mean():.2f}%")

## 9. Glosarium istilah teknis (subbab 3.3.3)

Satu baris per **bentuk permukaan**, sehingga ketentuan nomor 2 terpenuhi: satu entri dapat memiliki beberapa bentuk, misalnya `database` dan `basis data`, dengan penilaian dijatuhkan pada bentuk yang diucapkan.

Kolom `tanggal_kunci` adalah bukti kepatuhan pada ketentuan nomor 3, bahwa susunan entri dikunci sebelum koreksi transkrip acuan rampung.

In [ ]:
GLOSARIUM_PATH = OUT / "glosarium.csv"

if not GLOSARIUM_PATH.exists():
    contoh = [
        # mata_kuliah, entri_id, bentuk_kanonik, bentuk_permukaan, sumber, tanggal_kunci
        ("APS", "APS-001", "use case diagram", "use case diagram", "buku pedoman kurikulum 2024", ""),
        ("APS", "APS-001", "use case diagram", "diagram use case",  "materi ajar", ""),
        ("IMK", "IMK-001", "usability",        "usability",         "buku pedoman kurikulum 2024", ""),
        ("IMK", "IMK-001", "usability",        "kebergunaan",       "materi ajar", ""),
        ("AOK", "AOK-001", "pipeline",         "pipeline",          "buku pedoman kurikulum 2024", ""),
    ]
    pd.DataFrame(contoh, columns=["mata_kuliah", "entri_id", "bentuk_kanonik",
                                  "bentuk_permukaan", "sumber", "tanggal_kunci"]
                 ).to_csv(GLOSARIUM_PATH, index=False, encoding="utf-8-sig")
    print("Templat glosarium dibuat:", GLOSARIUM_PATH)
else:
    print("Glosarium sudah ada, tidak ditimpa.")

glosarium = pd.read_csv(GLOSARIUM_PATH, dtype=str, keep_default_na=False)
print(glosarium.groupby("mata_kuliah").agg(entri=("entri_id", "nunique"),
                                           bentuk=("bentuk_permukaan", "count")))

In [ ]:
# Pemeriksaan konsistensi ejaan istilah pada transkrip acuan.
# Ini pemeriksaan lintasan kedua: tidak butuh audio, cocok dikerjakan di TSV.
if semua:
    laporan = []
    for mk, g_glos in glosarium.groupby("mata_kuliah"):
        teks_mk = " ".join(master.loc[master["mata_kuliah"] == mk, "teks_acuan"].astype(str))
        teks_l = teks_mk.lower()
        for _, r in g_glos.iterrows():
            bentuk = r["bentuk_permukaan"].strip()
            if not bentuk:
                continue
            n = len(re.findall(r"(?<!\w)" + re.escape(bentuk.lower()) + r"(?!\w)", teks_l))
            # variasi kapitalisasi yang benar-benar muncul
            varian = sorted(set(re.findall(r"(?<!\w)" + re.escape(bentuk) + r"(?!\w)",
                                           teks_mk, flags=re.I)))
            laporan.append({"mata_kuliah": mk, "entri_id": r["entri_id"],
                            "bentuk_permukaan": bentuk, "kemunculan": n,
                            "varian_kapitalisasi": " | ".join(varian[:6]),
                            "jumlah_varian": len(varian)})

    df_glos = pd.DataFrame(laporan).sort_values(["mata_kuliah", "kemunculan"],
                                                ascending=[True, False])
    df_glos.to_csv(OUT / "cek_glosarium.csv", index=False, encoding="utf-8-sig")
    print("Entri yang tidak pernah muncul di transkrip acuan:")
    print(df_glos[df_glos["kemunculan"] == 0][["mata_kuliah", "bentuk_permukaan"]].to_string(index=False))
    print("\nEntri dengan kapitalisasi tidak seragam:")
    print(df_glos[df_glos["jumlah_varian"] > 1][
        ["mata_kuliah", "bentuk_permukaan", "varian_kapitalisasi"]].to_string(index=False))

## 10. Ringkasan untuk Bab 4

Jumlah rekaman, durasi, dan sebarannya per mata kuliah, sebagaimana dijanjikan pada akhir subbab 3.3.1.

In [ ]:
if semua:
    m = manifest.copy()
    m["durasi_potong_detik"] = pd.to_numeric(m["durasi_potong_detik"], errors="coerce")

    rk = (m.groupby(["mata_kuliah", "nama_mata_kuliah"])
            .agg(jumlah_rekaman=("file_id", "count"),
                 total_detik=("durasi_potong_detik", "sum"),
                 rerata_detik=("durasi_potong_detik", "mean"),
                 terpendek=("durasi_potong_detik", "min"),
                 terpanjang=("durasi_potong_detik", "max"))
            .reset_index())
    rk["total_jam"]    = (rk["total_detik"] / 3600).round(2)
    rk["rerata_menit"] = (rk["rerata_detik"] / 60).round(1)

    seg = (master.assign(dur=lambda d: d["dur"].astype(float),
                         n_kata=lambda d: d["teks_acuan"].astype(str).str.split().str.len())
                 .groupby("mata_kuliah")
                 .agg(jumlah_segmen=("seg_id", "count"),
                      rerata_dur_segmen=("dur", "mean"),
                      jumlah_kata=("n_kata", "sum")))
    seg["rerata_dur_segmen"] = seg["rerata_dur_segmen"].round(2)

    tabel = rk.merge(seg, on="mata_kuliah")[
        ["mata_kuliah", "nama_mata_kuliah", "jumlah_rekaman", "total_jam",
         "rerata_menit", "jumlah_segmen", "rerata_dur_segmen", "jumlah_kata"]]
    tabel.to_csv(OUT / "ringkasan_data.csv", index=False, encoding="utf-8-sig")
    print(tabel.to_string(index=False))
    print(f"\nTOTAL: {tabel['jumlah_rekaman'].sum()} rekaman, "
          f"{tabel['total_jam'].sum():.2f} jam, "
          f"{tabel['jumlah_segmen'].sum()} segmen, "
          f"{tabel['jumlah_kata'].sum()} kata")

In [ ]:
# Metadata run. Struktur bertingkat, sehingga disimpan sebagai JSON, bukan CSV.
import platform, sys
meta = {
    "tanggal": datetime.datetime.now().isoformat(timespec="seconds"),
    "tahap": "penyusunan transkrip acuan",
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "konfigurasi": CFG,
    "ambang_prioritas_simak": AMBANG,
    "versi_pustaka": {},
}
for nama in ["faster_whisper", "ctranslate2", "pandas", "numpy", "onnxruntime"]:
    try:
        meta["versi_pustaka"][nama] = __import__(nama).__version__
    except Exception:
        meta["versi_pustaka"][nama] = None

(OUT / "run_meta_transkrip_acuan.json").write_text(
    json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(meta["versi_pustaka"], indent=2))

---

## Keluaran

| Berkas | Isi |
|---|---|
| `data/raw/audio_src/*.m4a` | audio unduhan, apa adanya |
| `data/raw/json/*.json` | keluaran large-v3, arsip, tidak disunting |
| `data/processed/audio/*.wav` | 16 kHz mono PCM 16-bit, ternormalisasi, terpotong |
| `data/processed/srt/*.srt` | bahan koreksi |
| `data/processed/srt_done/*.srt` | hasil koreksi manual |
| `data/output/master/*.tsv` | **master transkrip acuan per rekaman** |
| `data/output/transkrip_acuan_gabungan.tsv` | master gabungan |
| `data/output/manifest_rekaman.csv` | daftar rekaman, durasi, penyaringan, split |
| `data/output/glosarium.csv` | glosarium per mata kuliah |
| `data/output/review_priority.csv` | segmen kandidat halusinasi |
| `data/output/laporan_validasi.csv` | hasil pemeriksaan struktural |
| `data/output/ringkasan_koreksi.csv` | proporsi kata yang dikoreksi |
| `data/output/ringkasan_data.csv` | tabel untuk Bab 4 |
| `data/output/run_meta_transkrip_acuan.json` | versi pustaka dan parameter |

## Membuka TSV di Excel tanpa merusak data

Jangan klik ganda. Gunakan `Data → From Text/CSV`, pilih **UTF-8**, pemisah **Tab**, lalu pada langkah pratinjau tetapkan seluruh kolom sebagai **Text**. Sebelum menempel apa pun, matikan `File → Options → Proofing → AutoCorrect Options`: kapitalisasi otomatis akan mengubah `eee` menjadi `Eee` dan melanggar pedoman verbatim.

Bila menyunting lewat Excel, simpan kembali sebagai **Unicode Text (.txt)** lalu ganti namanya, atau lebih aman sunting lewat pandas.

## Langkah berikutnya

Master ini menjadi masukan bagi tahap yang belum dicakup notebook ini: pembagian data 65 : 10 : 25 distratifikasi per mata kuliah (3.6.2), praproses teks dan penandaan glosarium (3.6.3), lalu Tahap 1 sampai Tahap 3 (3.6.5–3.6.7).